In [ ]:
### Imports + Config
import json, re
from pathlib import Path
from itertools import permutations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

repo_root_env = os.getenv("REPO_ROOT")
assert repo_root_env, "REPO_ROOT ist nicht gesetzt in .env"
REPO_ROOT = Path(repo_root_env).expanduser().resolve()
assert REPO_ROOT.exists(), f"REPO_ROOT existiert nicht: {REPO_ROOT}"

RESULTS_DIR = REPO_ROOT / "results"

### Ausgegebenes und annotiertes Label-Muster als Regex
SPEAKER_PATTERN = re.compile(r'\[Sprecher (\d+)\]:\s*')

In [ ]:
### Parser
# Parst '[Sprecher N]: text' Format.
# Returns: (tokens, labels) — gleich lang, ein Label pro Wort.
def parse_dialogue(text: str) -> tuple[list[str], list[str]]:

    tokens, labels = [], []
    for line in text.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        m = SPEAKER_PATTERN.match(line)
        if m:
            speaker = f"S{m.group(1)}"
            content = SPEAKER_PATTERN.sub('', line).strip()
        else:
            # Zeile ohne Speaker-Label → vorherigen Speaker fortsetzen
            speaker = labels[-1] if labels else "S0"
            content = line
        
        words = content.split()
        tokens.extend(words)
        labels.extend([speaker] * len(words))
    
    return tokens, labels

In [ ]:
### Prüfung, ob LLM Tokenstream verändert hat

def validate_token_stream(ref_text: str, hyp_text: str) -> dict:

    # Speaker-Labels entfernen
    ref_clean = SPEAKER_PATTERN.sub('', ref_text).strip().split()
    hyp_clean = SPEAKER_PATTERN.sub('', hyp_text).strip().split()

    if ref_clean == hyp_clean:
        return {"valid": True, "n_tokens": len(ref_clean)}

    from difflib import SequenceMatcher
    sm = SequenceMatcher(None, ref_clean, hyp_clean)
    ins, dels, subs = 0, 0, 0

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'insert':
            ins += j2 - j1
        elif tag == 'delete':
            dels += i2 - i1
        elif tag == 'replace':
            subs += max(i2 - i1, j2 - j1)

    return {
        "valid": False,
        "n_ref": len(ref_clean),
        "n_hyp": len(hyp_clean),
        "insertions": ins,
        "deletions": dels,
        "substitutions": subs,
        "token_error_rate": round((ins + dels + subs) / max(len(ref_clean), 1), 4),
    }

In [ ]:
### Permutationen beachten
def best_speaker_mapping(pred_labels: list[str], ref_labels: list[str]) -> dict:
    """
    Findet optimale 1:1-Zuordnung von pred→ref Speakern via Brute-Force.
    Funktioniert für ≤4 Sprecher (4! = 24 Permutationen).
    Returns: mapping dict {pred_speaker: ref_speaker}
    """
    pred_speakers = sorted(set(pred_labels))
    ref_speakers = sorted(set(ref_labels))
    
    best_mapping = None
    best_correct = -1
    
    for perm in permutations(ref_speakers):
        # Nur so viele wie pred_speakers hat
        if len(perm) < len(pred_speakers):
            continue
        mapping = {p: r for p, r in zip(pred_speakers, perm[:len(pred_speakers)])}
        correct = sum(1 for p, r in zip(pred_labels, ref_labels)
                      if mapping.get(p) == r)
        if correct > best_correct:
            best_correct = correct
            best_mapping = mapping
    
    return best_mapping

In [ ]:
### WSER berechnen
def compute_wser(pred_text: str, ref_text: str) -> dict:

    pred_tokens, pred_labels = parse_dialogue(pred_text)
    ref_tokens, ref_labels = parse_dialogue(ref_text)
    
    # Token-Stream muss identisch sein
    assert pred_tokens == ref_tokens, \
        f"Token mismatch: {len(pred_tokens)} vs {len(ref_tokens)} tokens \n Pred: {pred_tokens} \n Ref: {ref_tokens}"
        
    
    mapping = best_speaker_mapping(pred_labels, ref_labels)
    mapped_pred = [mapping.get(l, l) for l in pred_labels]
    
    n_total = len(ref_labels)
    n_correct = sum(1 for p, r in zip(mapped_pred, ref_labels) if p == r)
    n_errors = n_total - n_correct
    
    return {
        "wser": round(n_errors / n_total, 4) if n_total > 0 else 0.0,
        "n_total": n_total,
        "n_correct": n_correct,
        "n_errors": n_errors,
        "mapping": mapping,
        "n_speakers_pred": len(set(pred_labels)),
        "n_speakers_ref": len(set(ref_labels)),
    }

In [ ]:
### Boundary-WSER berechnen (im Fenster +- k Wörter um Sprecherwechsel)
def compute_boundary_wser(pred_text: str, ref_text: str, k: int = 2) -> dict:

    pred_tokens, pred_labels = parse_dialogue(pred_text)
    ref_tokens, ref_labels = parse_dialogue(ref_text)
    assert pred_tokens == ref_tokens
    
    mapping = best_speaker_mapping(pred_labels, ref_labels)
    mapped_pred = [mapping.get(l, l) for l in pred_labels]
    
    # Positionen der Sprecherwechsel in der Referenz finden
    boundary_positions = set()
    for i in range(1, len(ref_labels)):
        if ref_labels[i] != ref_labels[i-1]:
            # Fenster +-k um den Wechsel
            for j in range(max(0, i - k), min(len(ref_labels), i + k + 1)):
                boundary_positions.add(j)
    
    if not boundary_positions:
        return {"boundary_wser": 0.0, "n_boundary_words": 0}
    
    n_boundary = len(boundary_positions)
    n_correct = sum(1 for i in boundary_positions if mapped_pred[i] == ref_labels[i])
    
    return {
        "boundary_wser": round((n_boundary - n_correct) / n_boundary, 4),
        "n_boundary_words": n_boundary,
        "n_boundary_correct": n_correct,
        "n_turns": sum(1 for i in range(1, len(ref_labels)) if ref_labels[i] != ref_labels[i-1]),
    }

In [ ]:
### RTF-Berechnung aus meta.json
def compute_rtf(meta: dict, run_key: str) -> dict:

    audio_dur = meta.get("audio_duration_s")
    if not audio_dur or audio_dur == 0:
        return {"rtf_total": None, "rtf_post_asr": None}
    
    run = meta["runs"].get(run_key, {})
    sec_e2e = run.get("seconds_e2e", 0)
    sec_post = run.get("seconds_post_asr", 0)
    
    return {
        "rtf_total": round(sec_e2e / audio_dur, 4),
        "rtf_post_asr": round(sec_post / audio_dur, 4),
    }

In [ ]:
### Batch-Aggregate
rows = []

for result_dir in sorted(RESULTS_DIR.iterdir()):
    if not result_dir.is_dir():
        continue
    
    meta_path = result_dir / "meta.json"
    #DEBUG
    ref_path = result_dir / "Pyannote_diarized_segment_known.txt" 
    #DEBUGENDE
    #ref_path = result_dir / "diarization_ref.txt" 
    transcript_path = result_dir / "transcript.txt"
    
    if not all(p.exists() for p in [meta_path, ref_path, transcript_path]):
        print(f"  SKIP {result_dir.name}: Dateien fehlen")
        continue
    
    meta = json.loads(meta_path.read_text())
    ref_text = ref_path.read_text()
    transcript = transcript_path.read_text()
    known_n = meta.get("known_num_speakers")
    audio_dur = meta.get("audio_duration_s")
    
    # Sprecherzahl aus Dateiname (*_2S, *_3S, *_4S)
    spk_match = re.search(r'_(\d)S', result_dir.name)
    n_speakers = int(spk_match.group(1)) if spk_match else None
    
    # ASR-Zeit
    sec_asr = meta["seconds"].get("asr", meta["seconds"].get("asr", 0.0))
    
    # Alle Runs durchgehen
    for run_key, run_info in meta.get("runs", {}).items():
        # Output-Datei finden
        output_path = result_dir / f"{run_key}.txt"
        if not output_path.exists():
            continue
        
        hyp_text = output_path.read_text()
        if not hyp_text.strip():
            continue
        
        # Pipeline + Condition aus run_key parsen
        is_llm = run_key.startswith("LLM")
        condition = "unknown" if "unknown" in run_key else "known"
        
        if "seg" in run_key.lower():
            pipeline = "B"
        elif "word_aligned" in run_key.lower() or "word_wx" in run_key.lower(): # Fixen in _aligned in beiden Dateien!
            pipeline = "D"
        elif ("diarized_word" in run_key.lower()) and not ("word_alligned" in run_key.lower()): #analog
            pipeline = "C"
        elif "medgemma" in run_key.lower():
            pipeline = "A1"
        elif "llama" in run_key.lower() or "Meta-Llama" in run_key:
            pipeline = "A2"
        else:
            pipeline = run_key  # fallback
        
        # Token-Validierung (für LLM-Pipelines relevant)
        validation = validate_token_stream(transcript, hyp_text)
        
        # WSER berechnen (nur wenn valide)
        wser_result = None
        bwser_result = None
        if validation["valid"]:
            try:
                wser_result = compute_wser(hyp_text, ref_text)
                bwser_result = compute_boundary_wser(hyp_text, ref_text, k=2)
            except AssertionError as e:
                print(f"  WSER-Fehler {result_dir.name}/{run_key}: {e}")
        
        # RTF
        rtf = compute_rtf(meta, run_key)
        
        rows.append({
            "file": result_dir.name,
            "pipeline": pipeline,
            "condition": condition,
            "n_speakers": n_speakers,
            "audio_duration_s": audio_dur,
            # Validierung
            "valid": validation["valid"],
            "token_error_rate": validation.get("token_error_rate", 0.0),
            # WSER
            "wser": wser_result["wser"] if wser_result else None,
            "n_words": wser_result["n_total"] if wser_result else None,
            "n_speakers_pred": wser_result["n_speakers_pred"] if wser_result else None,
            # Boundary-WSER
            "boundary_wser": bwser_result["boundary_wser"] if bwser_result else None,
            "n_boundary_words": bwser_result["n_boundary_words"] if bwser_result else None,
            "n_turns": bwser_result["n_turns"] if bwser_result else None,
            # RTF
            "rtf_total": rtf["rtf_total"],
            "rtf_post_asr": rtf["rtf_post_asr"],
            # Meta
            "ok": run_info.get("ok", False),
            "error": run_info.get("error"),
        })

df = pd.DataFrame(rows)
print(f"Gesamt: {len(df)} Runs, {df['valid'].sum()} valide, {df['file'].nunique()} Dateien")
print(f"\nValidity Rate pro Pipeline:")
print(df.groupby('pipeline')['valid'].mean().round(3))

In [ ]:
#### DEBUG ALL IN ONE ZELLE

import json
import re
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

# True  -> gegen B-Pseudoreferenz testen
# False -> gegen echte manuelle Referenz testen
USE_B_AS_REFERENCE = True

def parse_condition(run_key: str) -> str:
    rk = run_key.lower()
    if "unknown" in rk:
        return "unknown"
    elif "known" in rk:
        return "known"
    return "other"

def parse_pipeline(run_key: str) -> str:
    rk = run_key.lower()

    if "medgemma" in rk:
        return "A1"
    elif "llama" in rk:
        return "A2"
    elif "word_alligned" in rk or "word_aligned" in rk or "word_wx" in rk:
        return "D"
    elif "diarized_word" in rk:
        return "C"
    elif "seg" in rk:
        return "B"
    else:
        return run_key

rows = []

for result_dir in sorted(RESULTS_DIR.iterdir()):
    if not result_dir.is_dir():
        continue

    meta_path = result_dir / "meta.json"
    transcript_path = result_dir / "transcript.txt"
    ref_path = (
        result_dir / "Pyannote_diarized_segment_known.txt"
        if USE_B_AS_REFERENCE
        else result_dir / "diarization_ref.txt"
    )

    if not meta_path.exists():
        print(f"SKIP {result_dir.name}: meta.json fehlt")
        continue
    if not transcript_path.exists():
        print(f"SKIP {result_dir.name}: transcript.txt fehlt")
        continue
    if not ref_path.exists():
        print(f"SKIP {result_dir.name}: {ref_path.name} fehlt")
        continue

    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    transcript = transcript_path.read_text(encoding="utf-8")
    ref_text = ref_path.read_text(encoding="utf-8")

    spk_match = re.search(r'_(\d)S', result_dir.name)
    n_speakers = int(spk_match.group(1)) if spk_match else None

    try:
        ref_tokens, ref_labels = parse_dialogue(ref_text)
        ref_n_tokens = len(ref_tokens)
        ref_n_speakers = len(set(ref_labels))
    except Exception as e:
        ref_n_tokens = None
        ref_n_speakers = None
        print(f"REF PARSE ERROR {result_dir.name}: {e}")

    for run_key, run_info in meta.get("runs", {}).items():
        output_path = result_dir / f"{run_key}.txt"

        row = {
            "file": result_dir.name,
            "run_key": run_key,
            "pipeline": parse_pipeline(run_key),
            "condition": parse_condition(run_key),
            "ref_file": ref_path.name,
            "n_speakers_from_filename": n_speakers,
            "known_num_speakers_meta": meta.get("known_num_speakers"),
            "audio_duration_s": meta.get("audio_duration_s"),
            "output_exists": output_path.exists(),
            "nonempty": None,
            "valid": None,
            "n_ref_eval": None,
            "n_hyp_eval": None,
            "insertions": None,
            "deletions": None,
            "substitutions": None,
            "token_error_rate": None,
            "wser": None,
            "boundary_wser": None,
            "n_words": None,
            "n_boundary_words": None,
            "n_turns": None,
            "n_speakers_pred": None,
            "n_speakers_ref_eval": ref_n_speakers,
            "mapping": None,
            "ok": run_info.get("ok", False),
            "error": run_info.get("error"),
            "rtf_total": None,
            "rtf_post_asr": None,
        }

        try:
            rtf = compute_rtf(meta, run_key)
            row["rtf_total"] = rtf.get("rtf_total")
            row["rtf_post_asr"] = rtf.get("rtf_post_asr")
        except Exception as e:
            row["error"] = f"{row['error']} | RTF_ERROR: {e}" if row["error"] else f"RTF_ERROR: {e}"

        if not output_path.exists():
            rows.append(row)
            continue

        hyp_text = output_path.read_text(encoding="utf-8")
        row["nonempty"] = bool(hyp_text.strip())

        if not hyp_text.strip():
            rows.append(row)
            continue

        validation = validate_token_stream(transcript, hyp_text)
        row["valid"] = validation["valid"]
        row["n_ref_eval"] = validation.get("n_ref", validation.get("n_tokens"))
        row["n_hyp_eval"] = validation.get("n_hyp", validation.get("n_tokens"))
        row["insertions"] = validation.get("insertions")
        row["deletions"] = validation.get("deletions")
        row["substitutions"] = validation.get("substitutions")
        row["token_error_rate"] = validation.get("token_error_rate")

        if validation["valid"]:
            try:
                wser_result = compute_wser(hyp_text, ref_text)
                bwser_result = compute_boundary_wser(hyp_text, ref_text, k=2)

                row["wser"] = wser_result["wser"]
                row["n_words"] = wser_result["n_total"]
                row["n_speakers_pred"] = wser_result["n_speakers_pred"]
                row["n_speakers_ref_eval"] = wser_result["n_speakers_ref"]
                row["mapping"] = str(wser_result["mapping"])

                row["boundary_wser"] = bwser_result["boundary_wser"]
                row["n_boundary_words"] = bwser_result["n_boundary_words"]
                row["n_turns"] = bwser_result["n_turns"]

            except Exception as e:
                row["error"] = f"{row['error']} | WSER_ERROR: {e}" if row["error"] else f"WSER_ERROR: {e}"

        rows.append(row)

debug_df = pd.DataFrame(rows)

print(f"Gesamt-Runs: {len(debug_df)}")
print(f"Dateien: {debug_df['file'].nunique()}")
print(f"Referenzmodus: {'B pseudo-reference' if USE_B_AS_REFERENCE else 'manual diarization_ref'}")

print("\n1) Alle Runs")
display(
    debug_df[
        [
            "file", "run_key", "pipeline", "condition",
            "valid", "token_error_rate", "wser", "boundary_wser",
            "n_speakers_from_filename", "known_num_speakers_meta",
            "n_speakers_ref_eval", "n_speakers_pred",
            "rtf_total", "rtf_post_asr",
            "ok", "error"
        ]
    ].sort_values(["file", "pipeline", "condition", "run_key"]).reset_index(drop=True)
)

print("\n2) Validity nach Pipeline x Condition")
display(
    debug_df.groupby(["pipeline", "condition"])["valid"]
    .mean()
    .rename("valid_rate")
    .reset_index()
    .sort_values(["pipeline", "condition"])
    .reset_index(drop=True)
)

print("\n3) Invalid Runs")
display(
    debug_df[debug_df["valid"] != True][
        [
            "file", "run_key", "pipeline", "condition",
            "output_exists", "nonempty", "valid",
            "n_ref_eval", "n_hyp_eval",
            "insertions", "deletions", "substitutions",
            "token_error_rate", "error"
        ]
    ].sort_values(["file", "pipeline", "condition", "run_key"]).reset_index(drop=True)
)

print("\n4) Nur valide WSER-Runs")
display(
    debug_df[debug_df["valid"] == True][
        [
            "file", "run_key", "pipeline", "condition",
            "wser", "boundary_wser", "n_words",
            "n_speakers_pred", "n_speakers_ref_eval", "mapping"
        ]
    ].sort_values(["file", "pipeline", "condition", "run_key"]).reset_index(drop=True)
)

print("\n5) Pivot: Datei x Pipeline/Condition")
display(
    debug_df.pivot_table(
        index="file",
        columns=["pipeline", "condition"],
        values="run_key",
        aggfunc="count",
        fill_value=0
    )
)

DEBUG_DF = debug_df.copy()
print("\nDEBUG_DF ist gesetzt.")

In [ ]:
### Ergebnis-Tabellen
# Table 2: WSER nach Pipeline × Sprecherzahl × Condition
pivot = df[df['valid'] == True].groupby(
    ['pipeline', 'n_speakers', 'condition']
)['wser'].agg(['median', lambda x: x.quantile(0.75) - x.quantile(0.25)])

pivot.columns = ['median', 'iqr']
pivot['display'] = pivot.apply(lambda r: f"{r['median']:.2f} ({r['iqr']:.2f})", axis=1)
print(pivot['display'].unstack(['n_speakers', 'condition']))

# Table 3: Boundary-WSER (nur Pipeline C vs D)
bwser_pivot = df[df['pipeline'].isin(['C', 'D']) & df['valid']].groupby(
    ['pipeline', 'condition']
)[['wser', 'boundary_wser']].median().round(3)
print(bwser_pivot)

# Table 4: RTF
rtf_pivot = df.groupby(['pipeline', 'condition'])[['rtf_total', 'rtf_post_asr']].median().round(3)
print(rtf_pivot)

# Validity Rate (für Paper-Text)
validity = df[df['pipeline'].isin(['A1', 'A2'])].groupby('pipeline')['valid'].agg(['sum', 'count'])
validity['rate'] = (validity['sum'] / validity['count']).round(3)
print(validity)

In [ ]:
### WSER-Stripplot
plot_df = df[df['valid'] == True].copy()
colors = {'unknown': '#D32F2F', 'known': '#1565C0'}

fig, axes = plt.subplots(1, 3, figsize=(7, 3), sharey=True)
for i, n_spk in enumerate([2, 3, 4]):
    ax = axes[i]
    sub = plot_df[plot_df['n_speakers'] == n_spk]
    
    sns.stripplot(data=sub, x='pipeline', y='wser', hue='condition',
                  palette=colors, dodge=True, jitter=0.12, size=5,
                  alpha=0.7, ax=ax, legend=(i == 2))
    
    # Median-Linien
    for j, pip in enumerate(sub['pipeline'].unique()):
        for k, cond in enumerate(['unknown', 'known']):
            vals = sub[(sub['pipeline'] == pip) & (sub['condition'] == cond)]['wser']
            if len(vals) > 0:
                med = vals.median()
                x_pos = j + (k - 0.5) * 0.4
                ax.plot([x_pos - 0.1, x_pos + 0.1], [med, med],
                       color=colors[cond], linewidth=2.0)
    
    ax.set_title(f'{n_spk} Speakers', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('WSER' if i == 0 else '')

plt.tight_layout()
plt.savefig('fig2_wser_stripplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### RTF-Balken-Diagramm
# RTF-Komponenten aus meta.json extrahieren

fig, ax = plt.subplots(figsize=(6, 3.5))

pipelines = ['A1', 'A2', 'B', 'C', 'D']
labels = ['A₁: MedGemma', 'A₂: Llama-3', 'B: Pya-Seg', 'C: Pya-FW', 'D: Pya-WX']

# Mediane pro Pipeline berechnen (aus df)
rtf_data = df[df['condition'] == 'unknown'].groupby('pipeline')['rtf_total'].median()

# Für Stacked Bars: Komponenten aus meta.json aggregieren
# → Implementierung analog zu meta["seconds"]-Aufbau

ax.bar(range(len(pipelines)), [rtf_data.get(p, 0) for p in pipelines])
ax.set_xticks(range(len(pipelines)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('RTF')
plt.tight_layout()
plt.savefig('fig3_rtf_bars.png', dpi=300, bbox_inches='tight')
plt.show()